# 热镀锌卷序优化建模

这个 notebook 用一个小型、内联的合同卷表展示 OptAgent 如何清晰表达热镀锌线（CGL/HDG）的卷序优化问题。重点是建模声明和求解器能力，而不是生产系统的 SQLite 前后处理流程。

业务目标：在一批冷轧/热轧来料合同卷中，给热镀锌机组排出一个更平稳的生产顺序，降低规格跳跃、退火温度跳跃、锌层切换、薄规格切换、后处理切换、外板窗口破坏和换辊风险。

## 1. 表格式示例数据

真实项目里，热镀锌排程输入通常不是一个孤立列表，而是来自订单、材料、机组和初始计划等多张业务表。本示例用 Python 的 `list[dict]` 模拟这些表，既保持轻量，又能看清数据如何进入模型。

In [ ]:
from __future__ import annotations

from dataclasses import dataclass

import pandas as pd

from optagent import ExternalCallbackContext, ModelBuilder, GaConfig, solve


@dataclass(frozen=True)
class Coil:
    # 面向业务的订单标识。生产环境里通常就是计划员和产线操作员熟悉的合同号或生产订单号。
    order_id: str
    # 材料号用于回溯到具体来料卷或准发材料。
    material_id: str
    # 目标机组。这里固定为一条热镀锌线，真实场景可扩展为多机组分配和排序。
    line_id: str
    # 来料来源会影响编排：热轧来料通常更厚，在镀锌 campaign 中不宜和冷轧基板随意穿插。
    source: str
    # 钢种族可近似表达冶金属性和退火制度的相似性。
    grade: str
    # 锌层或镀层族是热镀锌/连续镀锌最核心的 campaign 维度。
    zinc_layer: str
    # 厚度、宽度和退火温度共同决定相邻卷过渡是否平稳。
    thickness: float
    width: float
    anneal_temp: float
    # 重量字段保留下来，因为真实计划经常会限制有效 campaign 重量；本简化示例暂时只关注顺序过渡。
    weight: float
    # 外板和薄规格是质量敏感块，计划员通常会避免把它们打散到序列各处。
    outer_panel: bool = False
    thin_gauge: bool = False
    # 后处理路线用于表示钝化、涂油、无处理等下游设置差异。
    post_process: str = "none"


### 1.1 机组表

机组表描述热镀锌线本身的能力边界。真实项目中，这类表通常来自产线主数据，用于判断订单是否能在该机组生产，以及后续是否需要扩展到多机组分配。

In [ ]:
# 机组表：描述可用热镀锌线的基础能力。示例暂不把上下限做成硬约束，但这些字段说明模型可扩展的方向。
line_table = [
    {"line_id": "HDG-C208", "line_name": "2#连续热镀锌线", "min_width": 900, "max_width": 1550, "max_active_weight": 260.0},
]

pd.DataFrame(line_table)

### 1.2 材料表

材料表记录每卷来料的物理和冶金属性。热镀锌排程最关心的是来料来源、钢种、厚度、宽度、退火目标温度和重量，这些字段会直接影响相邻卷过渡成本。

In [ ]:
# 材料表：一行是一卷可排材料，保留来料来源、钢种、规格、退火目标温度和重量等物性。
material_table = [
    {"material_id": "M001", "source": "cold_rolled", "grade": "CQ", "thickness": 0.62, "width": 1220, "anneal_temp": 805, "weight": 23.5},
    {"material_id": "M002", "source": "cold_rolled", "grade": "CQ", "thickness": 0.60, "width": 1210, "anneal_temp": 800, "weight": 22.0},
    {"material_id": "M003", "source": "cold_rolled", "grade": "DQ", "thickness": 0.58, "width": 1195, "anneal_temp": 795, "weight": 21.6},
    {"material_id": "M004", "source": "cold_rolled", "grade": "EDDQ", "thickness": 0.72, "width": 1320, "anneal_temp": 835, "weight": 24.2},
    {"material_id": "M005", "source": "cold_rolled", "grade": "EDDQ", "thickness": 0.70, "width": 1310, "anneal_temp": 832, "weight": 23.8},
    {"material_id": "M006", "source": "hot_rolled", "grade": "HSLA", "thickness": 1.35, "width": 1510, "anneal_temp": 870, "weight": 28.4},
    {"material_id": "M007", "source": "hot_rolled", "grade": "HSLA", "thickness": 1.42, "width": 1490, "anneal_temp": 875, "weight": 29.1},
    {"material_id": "M008", "source": "cold_rolled", "grade": "BH", "thickness": 0.78, "width": 1260, "anneal_temp": 820, "weight": 22.7},
    {"material_id": "M009", "source": "cold_rolled", "grade": "BH", "thickness": 0.80, "width": 1250, "anneal_temp": 822, "weight": 22.9},
    {"material_id": "M010", "source": "cold_rolled", "grade": "CQ", "thickness": 0.64, "width": 1230, "anneal_temp": 806, "weight": 23.1},
]

pd.DataFrame(material_table)

### 1.3 订单表

订单表把材料和客户/工艺要求关联起来。锌层、外板、薄规格和后处理路线并不只是展示字段，它们会进入目标函数，影响最终卷序。

In [ ]:
# 订单表：一行是一张待排生产订单，表达客户/合同对锌层、外板、后处理等工艺要求。
order_table = [
    {"order_id": "O1001", "material_id": "M001", "line_id": "HDG-C208", "zinc_layer": "GI80", "outer_panel": False, "thin_gauge": False, "post_process": "chromate"},
    {"order_id": "O1002", "material_id": "M002", "line_id": "HDG-C208", "zinc_layer": "GI80", "outer_panel": False, "thin_gauge": False, "post_process": "chromate"},
    {"order_id": "O1003", "material_id": "M003", "line_id": "HDG-C208", "zinc_layer": "GI80", "outer_panel": False, "thin_gauge": True, "post_process": "chromate"},
    {"order_id": "O1004", "material_id": "M004", "line_id": "HDG-C208", "zinc_layer": "GA45", "outer_panel": True, "thin_gauge": False, "post_process": "oiling"},
    {"order_id": "O1005", "material_id": "M005", "line_id": "HDG-C208", "zinc_layer": "GA45", "outer_panel": True, "thin_gauge": False, "post_process": "oiling"},
    {"order_id": "O1006", "material_id": "M006", "line_id": "HDG-C208", "zinc_layer": "GI120", "outer_panel": False, "thin_gauge": False, "post_process": "none"},
    {"order_id": "O1007", "material_id": "M007", "line_id": "HDG-C208", "zinc_layer": "GI120", "outer_panel": False, "thin_gauge": False, "post_process": "none"},
    {"order_id": "O1008", "material_id": "M008", "line_id": "HDG-C208", "zinc_layer": "GI60", "outer_panel": True, "thin_gauge": False, "post_process": "oiling"},
    {"order_id": "O1009", "material_id": "M009", "line_id": "HDG-C208", "zinc_layer": "GI60", "outer_panel": True, "thin_gauge": False, "post_process": "oiling"},
    {"order_id": "O1010", "material_id": "M010", "line_id": "HDG-C208", "zinc_layer": "GI80", "outer_panel": False, "thin_gauge": False, "post_process": "chromate"},
]

pd.DataFrame(order_table)

### 1.4 初始计划表

初始计划表代表人工计划或上游 APS 的草案。OptAgent 不必从随机顺序开始，而是可以从这张草案顺序出发做修补和优化，这更符合现场使用方式。

In [ ]:
# 初始计划表：模拟人工计划或上游 APS 给出的草案顺序。优化不是从零开始，而是从这张计划表修补。
incumbent_plan_table = [
    {"line_id": "HDG-C208", "sequence_no": 1, "order_id": "O1001"},
    {"line_id": "HDG-C208", "sequence_no": 2, "order_id": "O1004"},
    {"line_id": "HDG-C208", "sequence_no": 3, "order_id": "O1006"},
    {"line_id": "HDG-C208", "sequence_no": 4, "order_id": "O1002"},
    {"line_id": "HDG-C208", "sequence_no": 5, "order_id": "O1005"},
    {"line_id": "HDG-C208", "sequence_no": 6, "order_id": "O1007"},
    {"line_id": "HDG-C208", "sequence_no": 7, "order_id": "O1009"},
    {"line_id": "HDG-C208", "sequence_no": 8, "order_id": "O1003"},
    {"line_id": "HDG-C208", "sequence_no": 9, "order_id": "O1008"},
    {"line_id": "HDG-C208", "sequence_no": 10, "order_id": "O1010"},
]

pd.DataFrame(incumbent_plan_table).sort_values("sequence_no")

### 1.5 表连接为模型输入

求解器最终需要的是“待排序对象”和“初始排列”。下面这一步把订单表、材料表和初始计划表连接起来，生成模型使用的 `Coil` 对象列表和 `incumbent_sequence`。

In [ ]:

def build_coils_from_tables() -> tuple[list[Coil], list[int]]:
    # 用 material_id 把材料物性补到订单上；这一步相当于 APS/MES 前处理中的轻量 join。
    material_by_id = {row["material_id"]: row for row in material_table}
    coils_by_order_id: dict[str, Coil] = {}
    for order in order_table:
        material = material_by_id[order["material_id"]]
        coils_by_order_id[order["order_id"]] = Coil(
            order_id=order["order_id"],
            material_id=order["material_id"],
            line_id=order["line_id"],
            source=material["source"],
            grade=material["grade"],
            zinc_layer=order["zinc_layer"],
            thickness=material["thickness"],
            width=material["width"],
            anneal_temp=material["anneal_temp"],
            weight=material["weight"],
            outer_panel=order["outer_panel"],
            thin_gauge=order["thin_gauge"],
            post_process=order["post_process"],
        )

    # 固定输出顺序，确保 order_id 到序列下标的映射可复现。
    coils = [coils_by_order_id[row["order_id"]] for row in sorted(order_table, key=lambda item: item["order_id"])]
    index_by_order_id = {coil.order_id: index for index, coil in enumerate(coils)}
    incumbent_sequence = [
        index_by_order_id[row["order_id"]]
        for row in sorted(incumbent_plan_table, key=lambda item: item["sequence_no"])
    ]
    return coils, incumbent_sequence


coils, incumbent_sequence = build_coils_from_tables()

# 展示业务输入表和 join 后的模型输入。notebook 输出中可以逐表查看，交互感比单一列表更接近真实系统。
model_input_df = pd.DataFrame([coil.__dict__ for coil in coils])
model_input_df

## 2. 业务规则评分

下面的目标函数保持在业务语言层：相邻卷的宽度、厚度、退火温度越平滑越好；锌层、薄规格、后处理和外板块尽量连续；宽度上跳较大时增加换辊风险。这个函数就是传给 OptAgent 的黑箱评分函数。

In [ ]:
def transition_cost(prev: Coil, curr: Coil) -> dict[str, float]:
    # 每一对相邻卷都会产生一组成本。这样评分函数就贴近计划员的表达：
    # “把 curr 接在 prev 后面生产，到底有多不顺？”
    width_jump = abs(prev.width - curr.width)
    thickness_jump = abs(prev.thickness - curr.thickness)
    temp_jump = abs(prev.anneal_temp - curr.anneal_temp)
    # 宽度上跳通常比宽度下跳更敏感，因为大幅上跳可能带来辊系和板形控制风险。
    width_up_jump = max(0.0, curr.width - prev.width)

    return {
        # 平滑性成本做了尺度换算，使宽度、厚度和炉温跳跃能在同一张成本表里比较。
        # 这些是示例权重，不是某条产线已经标定过的正式参数。
        "smooth_width": width_jump * 0.35,
        "smooth_thickness": thickness_jump * 900.0,
        "smooth_temperature": temp_jump * 1.6,
        # 镀层族切换成本较高，因为锌锅、气刀和相关设定的稳定性是 HDG campaign 的核心关注点。
        "zinc_campaign_change": 420.0 if prev.zinc_layer != curr.zinc_layer else 0.0,
        # 钢种族切换近似表示退火制度和冶金属性的切换。
        "grade_family_change": 120.0 if prev.grade[:2] != curr.grade[:2] else 0.0,
        # 薄规格、后处理和外板标识用于鼓励形成连续块，方便操作和质量人员监控。
        "thin_gauge_change": 180.0 if prev.thin_gauge != curr.thin_gauge else 0.0,
        "post_process_change": 160.0 if prev.post_process != curr.post_process else 0.0,
        "outer_panel_break": 260.0 if prev.outer_panel != curr.outer_panel else 0.0,
        # 小幅宽度上跳允许存在；超过 30 mm 的部分再按幅度惩罚。
        "roller_risk": max(0.0, width_up_jump - 30.0) * 3.0,
    }


def sequence_breakdown(sequence: list[int]) -> dict[str, float]:
    # 每类规则成本单独保留，便于解释最终结果。计划员可以看出改进来自镀层连续、规格更平滑、
    # 外板块减少打断，还是其他规则之间的取舍。
    costs = {
        "smooth_width": 0.0,
        "smooth_thickness": 0.0,
        "smooth_temperature": 0.0,
        "zinc_campaign_change": 0.0,
        "grade_family_change": 0.0,
        "thin_gauge_change": 0.0,
        "post_process_change": 0.0,
        "outer_panel_break": 0.0,
        "roller_risk": 0.0,
    }
    # 沿着候选序列逐对累加相邻卷过渡成本。
    for left, right in zip(sequence, sequence[1:]):
        for name, value in transition_cost(coils[left], coils[right]).items():
            costs[name] += value
    return {name: round(value, 3) for name, value in costs.items()}


def galvanizing_sequence_cost(sequence: list[int]) -> float:
    # OptAgent 最小化一个标量目标。这里把标量定义为所有业务成本之和，同时保留 sequence_breakdown
    # 方便求解后做分项诊断。
    return round(sum(sequence_breakdown(sequence).values()), 3)


# baseline 让优化结果更具体：最终报告可以逐条规则对比优化序列和初始草案。
baseline_costs = sequence_breakdown(incumbent_sequence)
baseline_cost = galvanizing_sequence_cost(incumbent_sequence)
baseline_cost, baseline_costs

## 3. OptAgent 建模声明

建模只需要三步：声明一个 `sequence_var`，把业务评分函数接成 `external_call`，再声明最小化目标。求解器负责在排列空间里搜索更好的卷序。

In [ ]:
# 1. 创建 OptAgent 模型。metadata 不是必需项，但当 notebook 扩展成可重复实验或 benchmark 时很有用。
builder = ModelBuilder(metadata={"case": "hot_dip_galvanizing_coil_sequence"})

# 2. 声明决策变量：合同卷下标的一个排列。default 使用初始草案，因此搜索从真实可理解的计划开始。
sequence = builder.sequence_var(
    size=len(coils),
    default=incumbent_sequence,
    name="galvanizing_coil_sequence",
)


def galvanizing_rule_cost(ctx: ExternalCallbackContext) -> float:
    order = [int(index) for index in ctx.value(sequence)]
    return galvanizing_sequence_cost(order)


# 3. 直接接入业务评分函数。external_call 让评分逻辑保持普通 Python 写法，排列搜索交给 OptAgent 处理。
builder.minimize(
    builder.external_call(galvanizing_rule_cost, name="galvanizing_rule_cost"),
    name="minimize_transition_and_campaign_cost",
)
# freeze 会把 builder 中的声明固化成不可变 program，后续交给 solve-first 策略 API。
program = builder.freeze()

program.metadata


## 4. 求解

这里使用 tabu 启发式做快速局部搜索。对示例读者而言，重点是：不需要手写邻域搜索、禁忌表或接受准则，只需要把卷序变量和业务评分交给 OptAgent。

In [ ]:
# 当前公开策略使用 GaConfig；序列变异和局部改进由 GA 配置统一声明。
result = solve(
    program,
    strategy=GaConfig(max_iterations=80),
    max_iterations=80,
    seed=7,
)

# UnifiedSolution 按 node_id 存储变量值。这里转成普通 int，方便展示和序列化。
best_sequence = [int(index) for index in result.variable_values[sequence.node_id]]
best_cost = galvanizing_sequence_cost(best_sequence)
{
    "baseline_cost": baseline_cost,
    "best_cost": best_cost,
    "improvement": round(baseline_cost - best_cost, 3),
    "best_sequence": [coils[index].order_id for index in best_sequence],
}


## 5. 结果解释

输出不只看总分，还要看每类规则的变化。热镀锌计划员通常会检查：锌层族是否更集中、宽度是否尽量同向平滑、退火温度是否少跳变、外板块是否被打散、后处理段是否频繁切换。

In [ ]:
# 对最优序列重新计算分项规则成本。这张表用于判断优化方案在业务上是否合理。
best_costs = sequence_breakdown(best_sequence)
cost_comparison = []
for name in sorted(baseline_costs):
    # delta 为负表示该规则成本降低；delta 为正表示求解器为了其他目标，在这条规则上付出了更多代价。
    cost_comparison.append({
        "rule": name,
        "baseline": baseline_costs[name],
        "best": best_costs[name],
        "delta": round(best_costs[name] - baseline_costs[name], 3),
    })

pd.DataFrame(cost_comparison)

In [ ]:
def sequence_view(sequence: list[int]) -> list[dict[str, object]]:
    # 把优化得到的下标排列还原成合同卷属性表。这是计划员审批排程前更自然的查看格式。
    rows = []
    for pos, index in enumerate(sequence, start=1):
        coil = coils[index]
        rows.append({
            "pos": pos,
            "order_id": coil.order_id,
            "material_id": coil.material_id,
            "line_id": coil.line_id,
            "source": coil.source,
            "grade": coil.grade,
            "zinc_layer": coil.zinc_layer,
            "thickness": coil.thickness,
            "width": coil.width,
            "anneal_temp": coil.anneal_temp,
            "outer_panel": coil.outer_panel,
            "post_process": coil.post_process,
        })
    return rows


pd.DataFrame(sequence_view(best_sequence))